# Reproducing *Deep Learning for Discrete-Time Hedging in Incomplete Markets*
### Fecamp, Mikael & Warin (2019) — arXiv:1902.05287
**ML in Finance — ENSAE Paris · Group Project 2025–2026**

---

| | |
|---|---|
| **Authors** | Emma Maria Moncia, Michael Carlo, Lucia Criado del Rey |
| **Date** | April 2026 |
| **Framework** | PyTorch (original paper: TensorFlow) |

---

## What this notebook reproduces

This notebook is a complete, self-contained reproduction of the paper's core numerical results,
followed by Section 6.2 (alternative loss functions) and Section 7 (Pareto frontier with transaction costs).

| Section | Paper reference | Content |
|---|---|---|
| §1–§5 | §2, §3, §4.3 | GBM simulation, BS benchmark, batch normalisation |
| §6 | §4.1–4.2, Table 1 | Five architectures vs BS delta — **Table 1** |
| §7 | §6.1, Figure 5 | Loss curves during training — **Figure 5** |
| §8 | §6.1, Figures 6–8 | Delta paths vs BS delta — **Figures 6–8** |
| §9 | §6.1 | P&L distribution baseline |
| §10 | §6.2, Figures 9–11 | Alternative loss functions — **Figures 9–11** |
| §11 | §7, Eq. (22), Figure 12 | Pareto frontier with transaction costs — **Figure 12** |
| §12 | — | Save all results for extension notebooks |

---

## Paper targets (Table 1)

| Architecture | Paper MSE |
|---|---|
| Black-Scholes Δ (theoretical optimum) | 1.61e-05 |
| Feedforward basic [10,10,10] | 1.32e-04 |
| Feedforward merged [10,10,10] | 1.37e-04 |
| **Augmented LSTM 50 units [10,10,10]** | **1.73e-05** |


## §0 · Imports

In [ ]:
import sys, os, json, time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm

# ── Make src importable ───────────────────────────────────────────────
def find_root():
    for candidate in [os.getcwd(),
                      os.path.abspath(os.path.join(os.getcwd(), '..')),
                      os.path.abspath(os.path.join(os.getcwd(), '../..'))]:
        if os.path.isdir(os.path.join(candidate, 'src')):
            return candidate
    raise RuntimeError("Cannot find 'src' folder. Run from inside the project.")

ROOT = find_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.simulators    import simulate_gbm, compute_norm_stats, normalize_with
from src.payoffs       import call_payoff
from src.benchmarks    import bs_call_price, bs_delta, bs_hedge_mse
from src.architectures import FeedforwardBasic, FeedforwardMerged, AugmentedLSTM
from src.losses        import mse
from src.training      import train_model
from torch.quasirandom import SobolEngine

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print(f'Device : {DEVICE}')
print(f'ROOT   : {ROOT}')


## §1 · Paper parameters (Section 4.3, Table 1 caption)

All values are taken verbatim from Section 4.3 of the paper.
**Do not change these for the reproduction** — they are the exact values
that produce Table 1.


In [ ]:
# ── Market ────────────────────────────────────────────────────────────
S0, K  = 1.0, 1.0       # at-the-money
sigma  = 0.2             # annual volatility
mu     = 0.0             # zero drift → martingale / risk-neutral
r_rf   = 0.0             # risk-free rate (named r_rf to avoid variable collision)
T      = 1/12            # 1-month maturity
dt     = 1/365           # one trading day per step
N      = int(T/dt)       # ≈ 30 hedging dates

# ── Training (paper §4.3) ─────────────────────────────────────────────
BATCH_SIZE  = 50
LR          = 1e-3
N_ITER      = 20_000
N_NORM      = 100_000    # reference paths for batch-norm stats
N_TRAIN     = 50_000
N_EVAL      = 100_000

# ── Architecture (paper §4.3) ─────────────────────────────────────────
LSTM_HIDDEN = 50
FF_WIDTH    = 10
FF_LAYERS   = 3
LIQ         = float('inf')   # unconstrained for Table 1

print(f'N hedging dates : {N}')
print(f'Batch size      : {BATCH_SIZE}')
print(f'Iterations      : {N_ITER:,}')
print(f'LSTM hidden     : {LSTM_HIDDEN}')


## §2 · Simulate paths and fix batch-norm statistics

GBM discretisation: $S_{t+1} = S_t \exp[(\mu-\frac{1}{2}\sigma^2)\Delta t + \sigma\sqrt{\Delta t}\,Z_t]$.

Normalisation statistics are computed **once** on a 100k reference set and then **frozen**
for the entire training — as stated in Section 4.3 of the paper.


In [ ]:
train_paths = simulate_gbm(N_TRAIN, S0, mu, sigma, dt, N, device=DEVICE, seed=0)
eval_paths  = simulate_gbm(N_EVAL,  S0, mu, sigma, dt, N, device=DEVICE, seed=1)

# Fixed normalisation stats (paper: 100k reference paths, frozen)
ref_paths = simulate_gbm(N_NORM, S0, mu, sigma, dt, N, device=DEVICE, seed=99)
norm_mean, norm_std = compute_norm_stats(ref_paths)
del ref_paths

train_norm = normalize_with(train_paths, norm_mean, norm_std)
eval_norm  = normalize_with(eval_paths,  norm_mean, norm_std)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(train_paths[:100].cpu().T, alpha=0.3, lw=0.7)
axes[0].axhline(K, color='red', ls='--', label=f'K={K}')
axes[0].set(title='100 sample GBM paths', xlabel='Step j', ylabel='$S_j$')
axes[0].legend()
axes[1].hist(train_paths[:, -1].cpu().numpy(), bins=80, color='steelblue', edgecolor='none')
axes[1].axvline(K, color='red', ls='--')
axes[1].set(title='Terminal price $S_T$', xlabel='$S_T$')
plt.tight_layout(); plt.show()
print(f'E[S_T] ≈ {train_paths[:,-1].mean().item():.4f}  (should be ≈ {S0} with μ=0)')


## §3 · Black-Scholes benchmark

The BS delta $N(d_1)$ is the analytically optimal hedge in the frictionless continuous-time limit.
Its MSE on discrete paths gives the lower bound we are trying to approach.

**Critical**: the BS premium must be collected at $t=0$ so that $\mathbb{E}[Y_T] \approx 0$
and $\text{MSE} \approx \text{Var}(Y_T)$. Without this, $\text{MSE} \approx p_{BS}^2 \approx 5.3\times 10^{-4}$.


In [ ]:
bs_mse, bs_pnl, bs_prem = bs_hedge_mse(eval_paths, K, sigma, T, dt)
BS_PRICE = bs_call_price(S0, K, T, sigma)

print(f'BS premium (learned) : {bs_prem:.5f}')
print(f'BS price (analytical): {BS_PRICE:.5f}')
print(f'BS MSE               : {bs_mse:.4e}   (paper: 1.61e-05)')
ratio = bs_mse / 1.61e-5
print(f'Ratio to paper       : {ratio:.2f}×   ({"✓ match" if 0.5 < ratio < 2 else "✗ CHECK PARAMS"})')


## §4 · Neural network architectures

Three architecture families are compared (Section 4.1–4.2, Figures 1–4):

- **Feedforward basic**: $N$ independent networks, one per step — no memory
- **Feedforward merged**: one shared network with time as extra input — no memory  
- **Augmented LSTM**: LSTMCell (shared across steps) + feedforward head — **has memory**

The key equation is (13): $\Delta_{t_j} = \Delta_{t_{j-1}} + l \cdot \tanh(\hat{C}_j)$,
which cumulates position changes and enforces the liquidity constraint.


In [ ]:
def build_models():
    return {
        'FF basic [10,10,10]'      : FeedforwardBasic(N=N, d=1, width=10, n_layers=3),
        'FF basic [10,15,30]'      : FeedforwardBasic(N=N, d=1, width=30, n_layers=3),
        'FF merged [10,10,10]'     : FeedforwardMerged(N=N, d=1, width=10, n_layers=3),
        'FF merged [10,15,30]'     : FeedforwardMerged(N=N, d=1, width=30, n_layers=3),
        'Augmented LSTM [10,10,10]': AugmentedLSTM(
            N=N, d=1, hidden=LSTM_HIDDEN, ff_width=FF_WIDTH, ff_layers=FF_LAYERS),
    }

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

models = {name: m.to(DEVICE) for name, m in build_models().items()}
print(f'  {"Architecture":<32} {"Parameters":>10}')
print('-' * 44)
for name, m in models.items():
    print(f'  {name:<32} {count_params(m):>10,}')


## §5 · Training function — Algorithm 1

Mini-batch Adam gradient descent on the global loss $L(Y_T) = \mathbb{E}[Y_T^2]$.
The `loss_fn` argument makes this reusable for all loss criteria in §10.


In [ ]:
PAYOFF_FN = lambda p: call_payoff(p[..., -1, 0] if p.dim() == 3 else p[:, -1], K)

def train_one(name, model, loss_fn=mse):
    """Train a model with Algorithm 1. loss_fn is configurable for §10."""
    return train_model(
        model,
        train_paths_raw  = train_paths,
        train_paths_norm = train_norm,
        eval_paths_raw   = eval_paths,
        eval_paths_norm  = eval_norm,
        payoff_fn        = PAYOFF_FN,
        loss_fn          = loss_fn,
        liq              = LIQ,
        n_iter           = N_ITER,
        batch_size       = BATCH_SIZE,
        lr               = LR,
        label            = name,
    )

def evaluate(model):
    """Evaluate model → (mse, pnl_array, premium)."""
    model.eval()
    with torch.no_grad():
        out  = model(eval_paths, eval_norm, liq=LIQ)
        g    = PAYOFF_FN(eval_paths)
        pnl  = (out['pnl'] - g).cpu().numpy()
        prem = float(out['premium'].item())
    return float(np.mean(pnl**2)), pnl, prem

print('Training helpers defined ✓')
print(f'Training budget: {N_ITER:,} iters × {len(models)} models')


## §6 · Train all five architectures — Table 1 reproduction

**Expected runtime**: ~10–20 min per model on CPU, ~2 min on GPU.


In [ ]:
histories = {}
for name, model in models.items():
    histories[name] = train_one(name, model)
    print(f'{name:35s}  best test MSE = {histories[name]["best_metric"]:.4e}')


In [ ]:
# ── Compute final MSE on the full eval set ────────────────────────────
paper_mse = {
    'FF basic [10,10,10]'      : 1.32e-4,
    'FF basic [10,15,30]'      : 1.31e-4,
    'FF merged [10,10,10]'     : 1.37e-4,
    'FF merged [10,15,30]'     : 1.30e-4,
    'Augmented LSTM [10,10,10]': 1.73e-5,
}

results = {'BS delta': {'mse': bs_mse, 'paper': 1.61e-5, 'prem': bs_prem, 'pnl': bs_pnl}}
for name, model in models.items():
    m, pnl, prem = evaluate(model)
    results[name] = {'mse': m, 'paper': paper_mse[name], 'prem': prem, 'pnl': pnl}

print('=' * 76)
print(f'  TABLE 1 REPRODUCTION  —  BS call, N={N}, T={T:.3f}yr, σ={sigma}')
print('=' * 76)
print(f'  {"Method":<32}  {"Our MSE":>12}  {"Paper":>12}  {"Ratio":>7}')
print('-' * 76)
for name, r in results.items():
    ratio = r['mse'] / r['paper']
    tag   = '✓' if 0.3 < ratio < 3.0 else '⚠'
    print(f'  {name:<32}  {r["mse"]:>12.3e}  {r["paper"]:>12.3e}  {ratio:>6.2f}× {tag}')
print('=' * 76)
lstm_mse = results['Augmented LSTM [10,10,10]']['mse']
print(f'\nKey ratio  LSTM / BS Δ : {lstm_mse/bs_mse:.2f}×  (paper: 1.07×)')


## §7 · Loss curves — Figure 5 reproduction

In [ ]:
selected = [list(histories.items())[0],
            list(histories.items())[2],
            list(histories.items())[-1]]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
colors = plt.cm.tab10(np.linspace(0, 1, 3))

for ax, (name, h), c in zip(axes, selected, colors):
    ax.semilogy(h['train'], alpha=0.3, color=c, lw=0.6, label='train')
    ax.semilogy(h['test_iters'], h['test'], color=c, lw=2, marker='o', ms=3, label='test')
    ax.axhline(bs_mse, color='k', ls=':', lw=1, label=f'BS Δ ({bs_mse:.1e})')
    ax.set(title=name, xlabel='Iteration')
    ax.grid(alpha=0.3)

axes[0].set_ylabel('MSE (log scale)')
axes[-1].legend(fontsize=8)
plt.suptitle('Figure 5 — Training curves', y=1.02)
plt.tight_layout(); plt.show()


## §8 · Delta paths vs Black-Scholes delta — Figures 6–8

In [ ]:
def bs_deltas_on_path(path, K, sigma, T, dt):
    N_steps = len(path) - 1
    T_rem   = np.clip(np.arange(N_steps, 0, -1) * dt, 1e-10, None)
    return bs_delta(path[:-1], K, T_rem, sigma)

lstm_model = models['Augmented LSTM [10,10,10]']
lstm_model.eval()
N_SHOW = 5
with torch.no_grad():
    out_show = lstm_model(eval_paths[:N_SHOW], eval_norm[:N_SHOW], liq=LIQ)
lstm_deltas = out_show['delta'].squeeze(-1).cpu().numpy()

fig, axes = plt.subplots(1, N_SHOW, figsize=(3.2*N_SHOW, 3.5), sharey=True)
for i, ax in enumerate(axes):
    path_np = eval_paths[i].cpu().numpy()
    ax.plot(bs_deltas_on_path(path_np, K, sigma, T, dt),
            '--', color='gray', lw=2, label='BS Δ', zorder=5)
    ax.plot(lstm_deltas[i], '-o', color='crimson', ms=3, lw=1.5, label='LSTM Δ')
    ax.set(title=f'Path {i+1}  $S_T$={path_np[-1]:.2f}',
           xlabel='Step j', ylim=(-0.05, 1.1))
    if i == 0: ax.set_ylabel('$\Delta_j$'); ax.legend(fontsize=8)

plt.suptitle('LSTM delta vs Black-Scholes delta along sample paths', y=1.02)
plt.tight_layout(); plt.show()

with torch.no_grad():
    out_big  = lstm_model(eval_paths[:500], eval_norm[:500], liq=LIQ)
all_lstm = out_big['delta'].squeeze(-1).cpu().numpy()
all_bs   = np.array([bs_deltas_on_path(eval_paths[i].cpu().numpy(), K, sigma, T, dt)
                     for i in range(500)])
corr = np.corrcoef(all_bs.ravel(), all_lstm.ravel())[0, 1]
print(f'Corr(LSTM Δ, BS Δ) over 500 paths × {N} steps = {corr:.4f}')
print(f'(Should be > 0.95 for a good reproduction)')


## §9 · P&L distribution baseline

This distribution is the **MSE reference** that the alternative loss functions
in §10 will be compared against.


In [ ]:
pnl_mse_lstm = results['Augmented LSTM [10,10,10]']['pnl']

fig, ax = plt.subplots(figsize=(10, 4))
bins = np.linspace(-0.06, 0.06, 160)
ax.hist(bs_pnl,       bins=bins, alpha=0.45, density=True, label='BS Δ', color='gray')
ax.hist(pnl_mse_lstm, bins=bins, alpha=0.6,  density=True, label='Augmented LSTM (MSE)', color='crimson')
ax.set(xlabel='$Y_T = X_T^\Delta - g(S_T)$', ylabel='Density',
       title='Terminal hedging-error distribution — MSE baseline')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'  {"Method":<32}  {"Mean":>9}  {"Std":>9}  {"VaR 5%":>9}  {"CVaR 5%":>9}  {"Prem":>7}')
print('-' * 80)
for name, r in results.items():
    p   = r['pnl']
    q5  = np.percentile(p, 5)
    cv5 = p[p < q5].mean()
    print(f'  {name:<32}  {p.mean():>9.2e}  {p.std():>9.2e}  '
          f'{q5:>9.4f}  {cv5:>9.4f}  {r.get("prem", float("nan")):>7.4f}')
print(f'\nBS analytical price = {BS_PRICE:.4f}  (learned premiums should be ≈ this)')


## §10 · Alternative loss functions — Section 6.2 (Figures 9–11)

The paper shows that different loss functions produce qualitatively different P&L distributions.
Three criteria are compared:

| Loss | Equation | α | Key property |
|---|---|---|---|
| MSE | $\mathbb{E}[Y^2]$ | — | Symmetric — already trained above |
| Asymmetric | $\mathbb{E}[(1+\alpha)Y^2\mathbf{1}_{Y\leq0} + Y^2\mathbf{1}_{Y>0}]$ | 1.5 | Penalises losses more than gains |
| Moment 2/4 | $\mathbb{E}[Y^2\mathbf{1}_{Y\geq0}] + \alpha\mathbb{E}[Y^4\mathbf{1}_{Y\leq0}]$ | 1.0 | Targets heavy loss tails |

**Expected result**: left tail shrinks, right tail expands relative to MSE.


In [ ]:
# ── Equations (5) and (6) ─────────────────────────────────────────────
def asymmetric_loss(pnl, alpha=1.5):
    """Equation (5) — losses penalised (1+alpha)× more than gains."""
    return ((1 + alpha) * pnl.pow(2) * (pnl <= 0).float()
            +             pnl.pow(2) * (pnl >  0).float()).mean()

def moment2_4_loss(pnl, alpha=1.0):
    """Equation (6) — fourth-moment on losses targets tail risk."""
    return (pnl.pow(2) * (pnl >= 0).float()).mean() +            alpha * (pnl.pow(4) * (pnl <= 0).float()).mean()

print('Loss functions defined:')
print('  asymmetric_loss(pnl, alpha=1.5)  — Equation (5)')
print('  moment2_4_loss(pnl,  alpha=1.0)  — Equation (6)')


In [ ]:
# ── Train Asymmetric loss ─────────────────────────────────────────────
print(f'[{time.strftime("%H:%M:%S")}] Training Asymmetric loss (α=1.5)...')

lstm_asym = AugmentedLSTM(
    N=N, d=1, hidden=LSTM_HIDDEN, ff_width=FF_WIDTH, ff_layers=FF_LAYERS
).to(DEVICE)

hist_asym = train_one(
    'Augmented LSTM — Asymmetric (α=1.5)',
    lstm_asym,
    loss_fn=lambda p: asymmetric_loss(p, alpha=1.5)
)

mse_asym, pnl_asym, prem_asym = evaluate(lstm_asym)

print(f'\n[{time.strftime("%H:%M:%S")}] Asymmetric done')
print(f'  MSE     : {mse_asym:.4e}   (MSE model: {np.mean(pnl_mse_lstm**2):.4e})')
print(f'  VaR 95% : {np.percentile(pnl_asym, 5):.5f}  (MSE model: {np.percentile(pnl_mse_lstm,5):.5f})')
print(f'  CVaR 95%: {pnl_asym[pnl_asym < np.percentile(pnl_asym,5)].mean():.5f}')
print(f'  Premium : {prem_asym:.5f}   (should be ~{BS_PRICE:.4f})')


In [ ]:
# ── Train Moment 2/4 — warm start from MSE model ─────────────────────
# The fourth-power term E[Y^4 * 1_{Y<=0}] is ~10^{-11} when Y~0.003.
# Training from scratch produces near-zero gradients → degenerate solution.
# Warm-starting from the MSE model (which already hedges correctly)
# gives the optimiser a good initial point so M2M4 fine-tunes rather than relearning.
#
# Key-remapping is required because the saved weights use different
# layer names ('cell.*', 'head.*') vs the current class ('lstm_cell.*', 'ff_head.*').

print(f'[{time.strftime("%H:%M:%S")}] Loading MSE weights for Moment 2/4 warm start...')

OUT = os.path.join(ROOT, 'results_table1')
raw = torch.load(os.path.join(OUT, 'table1_augmented_lstm_10_10_10.pt'),
                 map_location=DEVICE, weights_only=False)

# Remap saved key names → current AugmentedLSTM attribute names
key_map = {
    'cell.weight_ih': 'lstm_cell.weight_ih',
    'cell.weight_hh': 'lstm_cell.weight_hh',
    'cell.bias_ih':   'lstm_cell.bias_ih',
    'cell.bias_hh':   'lstm_cell.bias_hh',
    'head.0.weight':  'ff_head.0.weight',
    'head.0.bias':    'ff_head.0.bias',
    'head.2.weight':  'ff_head.2.weight',
    'head.2.bias':    'ff_head.2.bias',
    'head.4.weight':  'ff_head.4.weight',
    'head.4.bias':    'ff_head.4.bias',
    'head.6.weight':  'ff_head.6.weight',
    'head.6.bias':    'ff_head.6.bias',
    'premium':        'premium',
}

remapped = {key_map[k]: v for k, v in raw.items() if k in key_map}

lstm_m2m4 = AugmentedLSTM(
    N=N, d=1, hidden=LSTM_HIDDEN, ff_width=FF_WIDTH, ff_layers=FF_LAYERS
).to(DEVICE)

missing = lstm_m2m4.load_state_dict(remapped, strict=False)
print(f'  Missing keys after remap : {missing.missing_keys}')
print(f'  Unexpected keys          : {missing.unexpected_keys}')

# Fix premium manually — the saved file has premium=0 due to a save-cell bug
with torch.no_grad():
    lstm_m2m4.premium.fill_(BS_PRICE)

print(f'  Premium set to BS price  : {lstm_m2m4.premium.item():.5f}')
print(f'  LSTM weight norm         : {lstm_m2m4.lstm_cell.weight_ih.norm().item():.4f}')


In [ ]:
# ── Fine-tune with Moment 2/4 loss ───────────────────────────────────
print(f'[{time.strftime("%H:%M:%S")}] Fine-tuning with Moment 2/4 (α=1.0)...')

hist_m2m4 = train_one(
    'Augmented LSTM — Moment2/4 warm start (α=1.0)',
    lstm_m2m4,
    loss_fn=lambda p: moment2_4_loss(p, alpha=1.0)
)

mse_m2m4, pnl_m2m4, prem_m2m4 = evaluate(lstm_m2m4)

print(f'\n[{time.strftime("%H:%M:%S")}] Moment2/4 done')
print(f'  MSE     : {mse_m2m4:.4e}   target: 2e-05 to 5e-05')
print(f'  VaR 95% : {np.percentile(pnl_m2m4, 5):.5f}  target: better than {np.percentile(pnl_mse_lstm,5):.5f}')
print(f'  CVaR 95%: {pnl_m2m4[pnl_m2m4 < np.percentile(pnl_m2m4,5)].mean():.5f}')
print(f'  Premium : {prem_m2m4:.5f}   target: ~{BS_PRICE:.4f}')


In [ ]:
# ── Figures 9–11: overlay distributions ──────────────────────────────
pnl_by_loss = {
    'MSE':        pnl_mse_lstm,
    'Asymmetric': pnl_asym,
    'Moment 2/4': pnl_m2m4,
}
linestyles = ['-', '--', ':']
colors_loss = ['steelblue', 'crimson', 'darkorange']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
zoom_configs = [
    ('Full P&L distribution',  None,    None),
    ('Left tail — losses',    -0.025,  -0.003),
    ('Right tail — gains',     0.003,   0.025),
]
for ax, (title, xmin, xmax) in zip(axes, zoom_configs):
    for (name, pnl), ls, c in zip(pnl_by_loss.items(), linestyles, colors_loss):
        ax.hist(pnl, bins=300, density=True, histtype='step',
                label=name, linestyle=ls, color=c, linewidth=1.8)
    ax.axvline(0, color='gray', lw=0.8, ls='--', alpha=0.5)
    ax.set(title=title, xlabel=r'$Y_T = X_T^\Delta - g(S_T)$', ylabel='Density')
    if xmin is not None: ax.set_xlim(xmin, xmax)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle(
    f'Figures 9–11 — P&L distributions under different loss criteria\n'
    f'Augmented LSTM · GBM call · S0=K={S0}, σ={sigma}, T={T:.3f}yr, N={N}',
    y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig9_11_pnl_distributions.pdf'), bbox_inches='tight')
plt.show()
print('Saved → fig9_11_pnl_distributions.pdf')


In [ ]:
# ── Quantitative table ────────────────────────────────────────────────
print()
print('=' * 72)
print('  SECTION 6.2 — Loss function comparison')
print(f'  GBM call · S0=K={S0}, σ={sigma}, T={T:.3f}yr · {N_EVAL:,} eval paths')
print('=' * 72)
print(f'  {"Loss":<18} {"MSE":>10} {"Mean":>10} {"Std":>10} {"VaR 95%":>10} {"CVaR 95%":>10}')
print('-' * 72)
for name, pnl in pnl_by_loss.items():
    var95  = np.percentile(pnl, 5)
    cvar95 = pnl[pnl < var95].mean()
    print(f'  {name:<18} {np.mean(pnl**2):>10.3e} {pnl.mean():>10.3e} '
          f'{pnl.std():>10.3e} {var95:>10.4f} {cvar95:>10.4f}')
var95_bs  = np.percentile(bs_pnl, 5)
print()
print(f'  {"BS delta (ref)":<18} {bs_mse:>10.3e} {bs_pnl.mean():>10.3e} '
      f'{bs_pnl.std():>10.3e} {var95_bs:>10.4f} '
      f'{bs_pnl[bs_pnl<var95_bs].mean():>10.4f}')
print('=' * 72)


## §11 · Pareto frontier with transaction costs — Section 7 (Figure 12)

The paper trains **one** model for all $\alpha \in [0,1]$ simultaneously by adding $\alpha$
as an extra input and sampling it from a Sobol quasi-random sequence.

Objective (Equation 22):
$$d_\alpha = (1-\alpha)\,\mathbb{E}[TC] + \alpha\,\sqrt{\mathbb{E}[Y_T^2]}, \quad \alpha \in [0,1]$$

Parameters: transaction cost $c=0.02$, liquidity $l=0.2$ (paper values).


In [ ]:
# ── Pareto configuration ──────────────────────────────────────────────
TC_COST      = 0.02
LIQ_PARETO   = 0.2
PARETO_ITER  = 50_000   # paper uses 100k; 50k gives a smooth frontier
PARETO_BATCH = 64
PARETO_LR    = 1e-3
PARETO_H     = 50
PARETO_FFW   = 50       # paper increases FF width for the Pareto section
PARETO_FFL   = 3

# Pareto paths — fresh simulation with different seeds
train_paths_p = simulate_gbm(50_000, S0, mu, sigma, dt, N, device=DEVICE, seed=10)
eval_paths_p  = simulate_gbm(100_000, S0, mu, sigma, dt, N, device=DEVICE, seed=11)
train_norm_p  = normalize_with(train_paths_p, norm_mean, norm_std)
eval_norm_p   = normalize_with(eval_paths_p,  norm_mean, norm_std)

def compute_tc(delta, tc_cost):
    """TC = Σ_j |Δ_{t_j} - Δ_{t_{j-1}}| * c"""
    if delta.dim() == 3: delta = delta.squeeze(-1)
    first = delta[:, :1].abs()
    rest  = torch.diff(delta, dim=1).abs()
    return (torch.cat([first, rest], dim=1) * tc_cost).sum(dim=1)

print(f'Pareto paths: train={train_paths_p.shape}, eval={eval_paths_p.shape}')


In [ ]:
# ── ParetoLSTM: AugmentedLSTM with α as 3rd input ─────────────────────
class ParetoLSTM(nn.Module):
    """
    LSTM cell (shared across time) + FF head.
    Input at each step: (S̃_t, t/T, α) — dimension 3.
    Forget gate bias = 1.0 (Jozefowicz 2015).
    """
    def __init__(self, N, hidden=PARETO_H, ff_width=PARETO_FFW, ff_layers=PARETO_FFL):
        super().__init__()
        self.N         = N
        self.lstm_cell = nn.LSTMCell(input_size=3, hidden_size=hidden)
        layers = []
        in_d   = hidden
        for _ in range(ff_layers):
            layers += [nn.Linear(in_d, ff_width), nn.ReLU()]
            in_d    = ff_width
        layers.append(nn.Linear(in_d, 1))
        self.ff_head = nn.Sequential(*layers)
        self.premium = nn.Parameter(torch.tensor(0.0))

        nn.init.xavier_uniform_(self.lstm_cell.weight_ih)
        nn.init.orthogonal_(self.lstm_cell.weight_hh)
        nn.init.zeros_(self.lstm_cell.bias_ih)
        nn.init.zeros_(self.lstm_cell.bias_hh)
        self.lstm_cell.bias_ih.data[hidden:2*hidden].fill_(1.0)
        self.lstm_cell.bias_hh.data[hidden:2*hidden].fill_(1.0)

    def forward(self, paths_norm, liq, alpha_val):
        batch, device = paths_norm.shape[0], paths_norm.device
        h_t = torch.zeros(batch, self.lstm_cell.hidden_size, device=device)
        c_t = torch.zeros(batch, self.lstm_cell.hidden_size, device=device)
        delta_prev = torch.zeros(batch, device=device)
        pnl, deltas = torch.zeros(batch, device=device), []

        for j in range(self.N):
            s_j    = paths_norm[:, j].unsqueeze(1)
            t_feat = torch.full((batch, 1), j / self.N, device=device)
            a_feat = torch.full((batch, 1), alpha_val, device=device)
            x_j    = torch.cat([s_j, t_feat, a_feat], dim=1)

            h_t, c_t  = self.lstm_cell(x_j, (h_t, c_t))
            C_hat      = self.ff_head(h_t).squeeze(1)
            delta_t    = delta_prev + liq * torch.tanh(C_hat)
            pnl       += delta_t * (paths_norm[:, j+1] - paths_norm[:, j])
            deltas.append(delta_t.unsqueeze(1))
            delta_prev = delta_t

        return {'pnl':   pnl + self.premium.expand(batch),
                'delta': torch.cat(deltas, dim=1)}

pareto_model = ParetoLSTM(N).to(DEVICE)
n_p = sum(p.numel() for p in pareto_model.parameters() if p.requires_grad)
print(f'ParetoLSTM | {n_p:,} parameters')


In [ ]:
# ── Pareto training loop ──────────────────────────────────────────────
print(f'[{time.strftime("%H:%M:%S")}] Starting Pareto training ({PARETO_ITER:,} iters)...')

optimizer = torch.optim.Adam(pareto_model.parameters(), lr=PARETO_LR)
sobol     = SobolEngine(dimension=1, scramble=True, seed=42)

for it in tqdm(range(PARETO_ITER), desc='Pareto training'):
    pareto_model.train()
    alpha_val  = float(sobol.draw(1).item())
    idx        = torch.randint(0, len(train_paths_p), (PARETO_BATCH,))
    batch_raw  = train_paths_p[idx]
    batch_norm = train_norm_p[idx]

    out   = pareto_model(batch_norm, liq=LIQ_PARETO, alpha_val=alpha_val)
    g_ST  = call_payoff(batch_raw[:, -1], K)
    pnl   = out['pnl'] - g_ST
    tc    = compute_tc(out['delta'], TC_COST)
    loss  = (1 - alpha_val) * tc.mean() + alpha_val * torch.sqrt(pnl.pow(2).mean())

    optimizer.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(pareto_model.parameters(), 1.0)
    optimizer.step()

    if (it + 1) % 10_000 == 0:
        print(f'  [{it+1:>6}] α={alpha_val:.3f}  loss={loss.item():.4e}')

print(f'[{time.strftime("%H:%M:%S")}] Pareto training done ✓')


In [ ]:
# ── Trace Pareto frontier at 20 values of α ───────────────────────────
alphas_pts = np.linspace(0.0, 1.0, 20)
pareto_pts = []

pareto_model.eval()
with torch.no_grad():
    for a in alphas_pts:
        out  = pareto_model(eval_norm_p, liq=LIQ_PARETO, alpha_val=float(a))
        g_ST = call_payoff(eval_paths_p[:, -1], K)
        pnl  = (out['pnl'] - g_ST).cpu().numpy()
        tc   = compute_tc(out['delta'], TC_COST).cpu().numpy()
        pareto_pts.append({'alpha': float(a),
                           'mse':   float(np.mean(pnl**2)),
                           'tc':    float(np.mean(tc))})

# Figure 12
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([p['mse'] for p in pareto_pts],
        [p['tc']  for p in pareto_pts],
        'o-', color='steelblue', lw=2, ms=6, label='Neural Pareto frontier')
for pt in pareto_pts:
    if abs(pt['alpha'] - round(pt['alpha'], 1)) < 0.03:
        ax.annotate(f'α={pt["alpha"]:.1f}', (pt['mse'], pt['tc']),
                    fontsize=8, xytext=(6, 4), textcoords='offset points')
ax.set(xlabel='MSE  (hedging variance)',
       ylabel='Mean transaction cost  $\mathbb{E}[TC]$',
       title=f'Figure 12 — Pareto frontier\nGBM call · c={TC_COST}, l={LIQ_PARETO}')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig12_pareto_frontier.pdf'), bbox_inches='tight')
plt.show()
print('Saved → fig12_pareto_frontier.pdf')

print()
print(f'  α=0.0 point: MSE={pareto_pts[0]["mse"]:.3e}  TC={pareto_pts[0]["tc"]:.5f}  (no hedge — low cost)')
print(f'  α=1.0 point: MSE={pareto_pts[-1]["mse"]:.3e}  TC={pareto_pts[-1]["tc"]:.5f}  (pure variance — high cost)')


## §12 · Save all results

Saves everything to `results_table1/` for use in the report and extension notebooks.


In [ ]:
def to_serializable(obj):
    if isinstance(obj, np.ndarray):       return obj.tolist()
    if isinstance(obj, (np.floating,)):   return float(obj)
    if isinstance(obj, (np.integer,)):    return int(obj)
    if isinstance(obj, torch.Tensor):     return obj.detach().cpu().tolist()
    return obj

def _clean(x):
    if isinstance(x, torch.Tensor):              return x.detach().cpu().tolist()
    if isinstance(x, np.ndarray):                return x.tolist()
    if isinstance(x, (np.floating, np.integer)): return x.item()
    if isinstance(x, (list, tuple)):             return [_clean(v) for v in x]
    if isinstance(x, dict):                      return {k: _clean(v) for k,v in x.items()}
    return x

OUT = os.path.join(ROOT, 'results_table1')
os.makedirs(OUT, exist_ok=True)

# Table 1 model weights
for name, model in models.items():
    slug = (name.lower().replace(' ','_').replace('[','').replace(']','').replace(',','_'))
    torch.save(model.state_dict(), os.path.join(OUT, f'table1_{slug}.pt'))

# Loss function model weights
torch.save(lstm_asym.state_dict(),    os.path.join(OUT, 'lstm_asymmetric.pt'))
torch.save(lstm_m2m4.state_dict(),    os.path.join(OUT, 'lstm_m2m4.pt'))
torch.save(pareto_model.state_dict(), os.path.join(OUT, 'lstm_pareto.pt'))

# P&L arrays
np.save(os.path.join(OUT, 'table1_pnl_bs.npy'), bs_pnl)
for name, r in results.items():
    if 'pnl' in r:
        slug = (name.lower().replace(' ','_').replace('[','').replace(']','').replace(',','_'))
        np.save(os.path.join(OUT, f'table1_pnl_{slug}.npy'), r['pnl'])
np.save(os.path.join(OUT, 'pnl_lstm_asymmetric.npy'), pnl_asym)
np.save(os.path.join(OUT, 'pnl_lstm_m2m4.npy'),       pnl_m2m4)

# Training histories
hist_all = {**histories,
            'Asymmetric': hist_asym,
            'Moment2/4':  hist_m2m4}
with open(os.path.join(OUT, 'table1_histories.json'), 'w') as f:
    json.dump({n: {'train': _clean(h['train']), 'test': _clean(h['test']),
                   'test_iters': _clean(h['test_iters']),
                   'best_metric': _clean(h.get('best_metric'))}
               for n, h in hist_all.items()}, f, indent=2, default=to_serializable)

# Pareto frontier
with open(os.path.join(OUT, 'pareto_frontier.json'), 'w') as f:
    json.dump({'config': {'tc_cost': TC_COST, 'liq': LIQ_PARETO,
                          'n_iter': PARETO_ITER},
               'frontier': pareto_pts}, f, indent=2, default=to_serializable)

# Full summary (for mates to load)
summary = {
    'params': {
        'S0': S0, 'K': K, 'sigma': sigma, 'mu': mu, 'r': 0.0,
        'T': T, 'dt': dt, 'N': N,
        'batch_size': BATCH_SIZE, 'lr': LR, 'n_iter': N_ITER,
        'lstm_hidden': LSTM_HIDDEN, 'ff_width': FF_WIDTH, 'ff_layers': FF_LAYERS,
    },
    'results': {name: {'our_mse': r['mse'], 'paper_mse': r['paper']}
                for name, r in results.items()},
    'loss_comparison': {
        name: {'mse': float(np.mean(pnl**2)),
               'var95': float(np.percentile(pnl, 5)),
               'cvar95': float(pnl[pnl < np.percentile(pnl,5)].mean())}
        for name, pnl in pnl_by_loss.items()
    },
    'norm_mean': norm_mean.squeeze().cpu().tolist(),
    'norm_std':  norm_std.squeeze().cpu().tolist(),
}
with open(os.path.join(OUT, 'table1_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=to_serializable)

print(f'Saved to {OUT}/')
print()
for fname in sorted(os.listdir(OUT)):
    size = os.path.getsize(os.path.join(OUT, fname))
    print(f'  {fname:<48} {size/1024:>7.1f} KB')


---
## Summary — what has been reproduced

| Paper element | Section | Status |
|---|---|---|
| GBM simulation | §2.1 | ✓ |
| Batch normalisation (fixed, 100k paths) | §4.3 | ✓ |
| Feedforward basic architecture | §4.1.1, Fig 1 | ✓ |
| Feedforward merged architecture | §4.1.2, Fig 2 | ✓ |
| Augmented LSTM (Eq. 13) | §4.2, Fig 4 | ✓ |
| MSE loss — Equation (4) | §2.2 | ✓ |
| Algorithm 1 — global training | §4, Alg 1 | ✓ |
| Table 1 reproduction | §4.3.1 | ✓ |
| Figure 5 — loss curves | §6.1 | ✓ |
| Figures 6–8 — delta paths | §6.1 | ✓ |
| Asymmetric loss — Equation (5) | §6.2 | ✓ |
| Moment 2/4 loss — Equation (6) | §6.2 | ✓ (warm start) |
| Figures 9–11 — P&L distributions | §6.2 | ✓ |
| Pareto frontier — Equation (22) | §7 | ✓ |
| Figure 12 — Pareto curve | §7 | ✓ |

## Handover for mates — loading instructions

```python
import torch, json, numpy as np

with open('../results_table1/table1_summary.json') as f:
    s = json.load(f)

# Restore parameters
S0, K, sigma, T, N, dt = s['params']['S0'], s['params']['K'], \
    s['params']['sigma'], s['params']['T'], s['params']['N'], s['params']['dt']

# Restore normalisation (use EXACTLY these — do not recompute)
norm_mean = torch.tensor(s['norm_mean'])
norm_std  = torch.tensor(s['norm_std'])

# Load baseline P&L arrays for comparison
pnl_mse_lstm = np.load('../results_table1/table1_pnl_augmented_lstm_10_10_10.npy')
pnl_asym     = np.load('../results_table1/pnl_lstm_asymmetric.npy')
bs_pnl       = np.load('../results_table1/table1_pnl_bs.npy')
```
